# TabPFN for Uplift Modeling with Meta-Learners (Hillstrom)

This notebook evaluates whether TabPFN can improve uplift modeling performance when used inside meta-learners compared to traditional models (LinearRegression, LightGBM) and a standalone causal model (CausalPFN).

**Dataset**: Hillstrom MineThatData — a randomized controlled trial from an email marketing campaign.
- 64,000 customers, randomly assigned equally to three arms: No Email (control), Men's Email, Women's Email
- Evaluated as two separate binary tasks, mirroring CausalPFN's benchmark setup:
  - **Hill(1)**: Men's Email vs. No Email
  - **Hill(2)**: Women's Email vs. No Email (Women's Email is known to have larger uplift)
- 10 covariates (recency, history, mens, womens, zip_code, newbie, channel, and 3 features parsed from history_segment)
- Outcome: `visit` (binary 0/1 — did the customer visit the website in the two weeks following the campaign)
- Each run uses one arm; switch `TREATMENT_ARM` in the data loading cell to change between Hill(1) and Hill(2)

**Evaluation protocol**:
- 20 repeated stratified train/test splits (80/20, stratified on T × Y) on the full dataset (~42,000 rows per arm)
- All models trained on training set only, evaluated on held-out test set
- R-learner (NonParamDML) and DR-learner (DRLearner) use 5-fold cross-fitting within the training data for nuisance estimation
- LightGBM hyperparameters tuned once per split via RandomizedSearchCV (30 iterations, 3-fold CV)

**Metrics** (all computed on the test set per split):
- **AUQC**: Area between the adjusted Qini curve and its diagonal (the random-model baseline), unitless. Expected value under a random CATE ranking is 0 by construction; significance test checks whether the 95% CI excludes zero.
- **Policy Gain @k%** (k = 10, 20, 30, 40): IPW policy value for treating the top-k% minus the random k% baseline, in visit rate units

**Models**:
- Meta-learners: S, T, X, R (NonParamDML, cv=5), DR (DRLearner, cv=5)
- Base models: LinearRegression, LightGBM (tuned), TabPFN, TabICL
- Standalone: CausalForestDML (cv=5, LightGBM nuisance), CausalPFN

## 1. Setup and Imports

In [1]:
# ── IMPORTS ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import scipy.stats as st
import pickle
from collections import defaultdict
from datetime import datetime
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold, KFold
from lightgbm import LGBMRegressor, LGBMClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
import tabpfn
from tabpfn import TabPFNRegressor, TabPFNClassifier
from tabicl import TabICLRegressor, TabICLClassifier
from econml.metalearners import SLearner, TLearner, XLearner
from econml.dml import NonParamDML, CausalForestDML
from econml.dr import DRLearner
import matplotlib.pyplot as plt
import warnings
import torch

from causalpfn import CATEEstimator
from causalpfn.evaluation import get_qini_curve as _get_qini_curve

def _compute_auqc(qini_curve):
    n = len(qini_curve)
    phi = np.linspace(1/n, 1.0, n)
    diagonal = phi * qini_curve[-1]
    return np.trapezoid(qini_curve - diagonal, phi)

def get_qini_curve(T, Y, cate):
    # _compute_auqc always computes the diagonal-subtracted area
    curve, _ = _get_qini_curve(T, Y, cate, normalize=False)
    return curve, _compute_auqc(curve)


# ── DEVICE DETECTION ─────────────────────────────────────────────────────────
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f"TabPFN {tabpfn.__version__} | device: {device}")

# CausalPFN does not support MPS
causalpfn_device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"CausalPFN device: {causalpfn_device}")

# TabICL does not support MPS — fall back to CPU
tabicl_device = "cpu" if device == "mps" else device
print(f"TabICL device: {tabicl_device}")


# ── EXTRA ─────────────────────────────────────────────────────────
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)  # no-op if no CUDA
warnings.filterwarnings("ignore")

import os
os.environ["TABPFN_NO_TELEMETRY_PROMPT"] = "1"  # must be set before tabpfn import
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

# Results output directory — timestamped to avoid overwriting previous runs
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
RESULTS_DIR = os.path.join('results', f'exp_04_Hill_{RUN_ID}')
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Results directory: {RESULTS_DIR}")

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TabPFN 7.1.1 | device: mps
CausalPFN device: cpu
TabICL device: cpu
Results directory: results/exp_04_Hill_20260422_145752


In [2]:
# ── Import TabPFN 2.5 alongside 2.6 ────────────────────────────────────────
import subprocess, sys as _sys

TABPFN25_DIR = "./tabpfn_v25_install"
os.makedirs(TABPFN25_DIR, exist_ok=True)

# Install tabpfn 2.5 to isolated directory if not already present
_tabpfn25_installed = any("tabpfn" in d for d in os.listdir(TABPFN25_DIR))
if not _tabpfn25_installed:
    print("Installing TabPFN 2.5 to isolated directory...")
    subprocess.check_call([
        _sys.executable, "-m", "pip", "install", "tabpfn==6.4.1",
        f"--target={TABPFN25_DIR}", "--quiet", "--no-deps"
    ])
    print("Done.")

# Save current tabpfn 2.6 module references
_tabpfn26_mods = {k: v for k, v in _sys.modules.items()
                  if k == "tabpfn" or k.startswith("tabpfn.")}

# Temporarily inject 2.5 path and load its classes
_sys.path.insert(0, TABPFN25_DIR)
for _k in [k for k in list(_sys.modules) if k == "tabpfn" or k.startswith("tabpfn.")]:
    del _sys.modules[_k]

import tabpfn as _tabpfn25
TabPFNRegressor25 = _tabpfn25.TabPFNRegressor
TabPFNClassifier25 = _tabpfn25.TabPFNClassifier
TABPFN25_VERSION = _tabpfn25.__version__

# Restore tabpfn 2.6
_sys.path.remove(TABPFN25_DIR)
for _k in [k for k in list(_sys.modules) if k == "tabpfn" or k.startswith("tabpfn.")]:
    del _sys.modules[_k]
_sys.modules.update(_tabpfn26_mods)

print(f"TabPFN 2.5 version: {TABPFN25_VERSION}")
print(f"TabPFN 2.6 version: {tabpfn.__version__}")

TabPFN 2.5 version: 6.4.1
TabPFN 2.6 version: 7.1.1


## 2. Helper Functions

LightGBM tuning via `RandomizedSearchCV` (30 iterations, 3-fold CV, 3 hyperparameters). IPW-based policy value estimation for top-k% targeting. Qini curve interpolation. AUQC is computed as the area between the adjusted Qini curve and its diagonal (the random-model baseline), so its expected value under a random CATE ranking is 0 by construction.

In [3]:
# ── TUNING ────────────────────────────────────────────────────
# LightGBM hyperparameter search space
LGBM_GRID = {
    'num_leaves':        [15, 31, 63],
    'min_child_samples': [20, 50, 100],
    #'learning_rate':     [0.01, 0.05, 0.1],
    'n_estimators':      [200, 500, 1000],
}

N_ITER_STRONG = 30   # RandomizedSearchCV draws
N_EST         = 1000  # base tree count (overridden by grid)
NUISANCE_CV   = 5    # K-fold cross-fitting for R- and DR-learner nuisance models

# LightGBM Tuning
def tune_lgbm(X, y, classifier=False, stratify=None, n_iter=N_ITER_STRONG, seed=42):
    """Tune LGBM via RandomizedSearchCV. Returns best_params_ dict.

    - classifier=False  → LGBMRegressor, scored by neg_MSE
    - classifier=True   → LGBMClassifier, scored by neg_log_loss
    - stratify          → use StratifiedKFold on this variable (regression only);
                          if None, falls back to KFold with adaptive n_splits
    """
    X = np.asarray(X)

    if classifier:
        # Propensity Model (classification)
        base    = LGBMClassifier(n_estimators=N_EST, random_state=seed, verbose=-1)
        scoring = 'neg_log_loss'
        cv      = list(StratifiedKFold(3, shuffle=True, random_state=seed).split(X, y))
    else:
        # Outcome Model (regression): S-, R-, and DR-learner
        base    = LGBMRegressor(n_estimators=N_EST, random_state=seed, verbose=-1)
        scoring = 'neg_mean_squared_error'
        if stratify is not None:
            cv  = list(StratifiedKFold(3, shuffle=True, random_state=seed).split(X, stratify))
        else:
            # Single Treatment Arm: T- and X-learner
            n_splits = min(3, max(2, len(y) // 20))
            cv  = list(KFold(n_splits, shuffle=True, random_state=seed).split(X))

    search = RandomizedSearchCV(
        base, LGBM_GRID, n_iter=n_iter, scoring=scoring,
        cv=cv, n_jobs=1, random_state=seed,
    )

    search.fit(X, y)
    return search.best_params_


# Pseudo-Outcome Tuning: X-, R-, and DR-learner final-stage models
def make_lgbm_final(seed=42):
    """LGBM wrapped in RandomizedSearchCV for final-stage tuning on pseudo-outcomes."""
    return RandomizedSearchCV(
        LGBMRegressor(n_estimators=N_EST, random_state=seed, verbose=-1),
        LGBM_GRID, n_iter=N_ITER_STRONG, cv=3, scoring='neg_mean_squared_error',
        n_jobs=1, random_state=seed,
    )


# ── Metrics ───────────────────────────────────────────────────
TOP_K_PERCENTAGES = [10, 20, 30, 40]

def calculate_topk_policy_metrics(y_true, treatment, predicted_ite,
                                   k_percentages=TOP_K_PERCENTAGES, min_treated=3, min_control=3):
    """IPW policy value for top-k% treatment policies (unbiased under RCT design)."""
    treatment = np.asarray(treatment)
    y_true = np.asarray(y_true)
    predicted_ite = np.asarray(predicted_ite).flatten()
    n = len(y_true)
    sorted_idx = np.argsort(-predicted_ite)
    e = treatment.mean()
    treat_all = (treatment * y_true / e).mean() if e > 0 else np.nan
    treat_none = ((1 - treatment) * y_true / (1 - e)).mean() if e < 1 else np.nan
    metrics = {}
    for k in k_percentages:
        n_top = int(np.ceil(n * k / 100))
        pi = np.zeros(n)
        pi[sorted_idx[:n_top]] = 1.0
        n_t_top = (treatment[sorted_idx[:n_top]] == 1).sum()
        n_c_bot = (treatment[sorted_idx[n_top:]] == 0).sum()
        if n_t_top < min_treated or n_c_bot < min_control:
            pv = np.nan
        else:
            pv = (pi * treatment * y_true / e + (1 - pi) * (1 - treatment) * y_true / (1 - e)).mean()
        rand_k = (k / 100) * treat_all + (1 - k / 100) * treat_none if not (np.isnan(treat_all) or np.isnan(treat_none)) else np.nan
        gain = pv - rand_k if not (np.isnan(pv) or np.isnan(rand_k)) else np.nan
        metrics[f'policy_value_{k}'] = pv
        metrics[f'random_policy_{k}'] = rand_k
        metrics[f'policy_gain_{k}'] = gain
    return metrics

def interpolate_qini_curve(qini_curve, n_grid=101):
    n = len(qini_curve)
    x_orig = np.concatenate([[0.0], np.linspace(1/n, 1.0, n)])
    y_orig = np.concatenate([[0.0], qini_curve])
    return np.interp(np.linspace(0.0, 1.0, n_grid), x_orig, y_orig)

## 3. Data Loading

Load the Hillstrom email marketing RCT dataset via `sklift`, apply feature preprocessing, and encode treatments. The full arm dataset (~42,000 rows) is used for each split.

In [ ]:
# --- Load Hillstrom data ---
from sklearn.preprocessing import LabelEncoder
from sklift.datasets import fetch_hillstrom

OUTCOME_COL   = "visit"       # "visit", "conversion", or "spend"
CONTROL_ARM   = "No E-Mail"
# Set TREATMENT_ARM to one of "Mens E-Mail" or "Womens E-Mail".
# Run the notebook separately for each arm, mirroring CausalPFN's Hill(1) / Hill(2) tasks.
TREATMENT_ARM = "Womens E-Mail"   # Hill(2): Women's Email vs. No Email
# TREATMENT_ARM = "Mens E-Mail"   # Hill(1): Men's Email vs. No Email

X_df, y_df, t_df = fetch_hillstrom(return_X_y_t=True, target_col=OUTCOME_COL)

# --- Feature preprocessing (mirrors CausalPFN benchmarks/hillstrom.py) ---
all_X = X_df.copy()
for col in ["zip_code", "channel"]:
    le = LabelEncoder()
    all_X[col] = le.fit_transform(all_X[col].astype(str))

def parse_history_segment(s):
    if "-" not in s:
        parts = s.split("+")
        item_id = int(parts[0].strip().split(") ")[0])
        left = float(parts[0].split("$")[1].replace(",", ""))
        right = left * 2
    else:
        parts = s.split(" - ")
        item_id = int(parts[0].split(") ")[0])
        left = float(parts[0].split("$")[1].replace(",", ""))
        right = float(parts[1].split("$")[1].replace(",", ""))
    return item_id, left, right

parsed = all_X["history_segment"].apply(parse_history_segment)
all_X["history_segment_id"]    = parsed.apply(lambda x: x[0])
all_X["history_segment_lower"] = parsed.apply(lambda x: x[1])
all_X["history_segment_upper"] = parsed.apply(lambda x: x[2])
all_X = all_X.drop(columns=["history_segment"])

# --- Treatment encoding: keep only control vs. chosen treatment arm ---
def encode_treatment(s):
    if s == CONTROL_ARM:   return 0
    if s == TREATMENT_ARM: return 1
    return -1

t_encoded = t_df.astype(str).map(encode_treatment).values
valid_rows = t_encoded >= 0

all_X_arr = all_X.values[valid_rows].astype(np.float32)
all_Y_arr = y_df.values[valid_rows].astype(np.float32).ravel()
all_T_arr = t_encoded[valid_rows].astype(int)

feature_cols = list(all_X.columns)

# --- Data summary ---
n_total   = len(all_T_arr)
n_treated = int(all_T_arr.sum())
n_control = n_total - n_treated
ate       = all_Y_arr[all_T_arr == 1].mean() - all_Y_arr[all_T_arr == 0].mean()

arm_tag = "Hill2" if TREATMENT_ARM == "Womens E-Mail" else "Hill1"
print("=" * 55)
print(f"Hillstrom Dataset Summary — {arm_tag} ({TREATMENT_ARM} vs. {CONTROL_ARM})")
print("=" * 55)
print(f"  Total samples  : {n_total}")
print(f"  Treated        : {n_treated}  ({100*n_treated/n_total:.1f}%)")
print(f"  Control        : {n_control}  ({100*n_control/n_total:.1f}%)")
print(f"  Features       : {feature_cols}")
print(f"  Outcome        : {OUTCOME_COL}  range [{all_Y_arr.min():.3g}, {all_Y_arr.max():.3g}]")
print(f"                   mean  {all_Y_arr.mean():.4f}  (std {all_Y_arr.std():.4f})")
print(f"  ATE (naive)    : {ate:+.4f}  (treated mean - control mean)")

# --- Configuration ---
# Set DRY_RUN = True for a quick end-to-end check (2 splits, fewer trees)
DRY_RUN = False

N_SPLITS      = 1    if DRY_RUN else 10
N_GRID        = 101

# --- Storage ---
results = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
qini_curves = defaultdict(lambda: defaultdict(list))

META_LEARNERS = ['S', 'T', 'X', 'R', 'DR', 'CF', 'CausalPFN']
BASE_MODELS = ['LinearRegression', 'LightGBM', 'TabPFN_v2.5','TabPFN_v2.6', 'TabICL']


print(f"\nConfiguration: {N_SPLITS} repeated splits (full dataset, 80/20), DRY_RUN={DRY_RUN}")

Hillstrom dataset: 100%|████████████████████████████████████████████████████████████| 443k/443k [00:00<00:00, 1.17MiB/s]

Hillstrom Dataset Summary — Hill2 (Womens E-Mail vs. No E-Mail)
  Total samples  : 42693
  Treated        : 21387  (50.1%)
  Control        : 21306  (49.9%)
  Features       : ['recency', 'history', 'mens', 'womens', 'zip_code', 'newbie', 'channel', 'history_segment_id', 'history_segment_lower', 'history_segment_upper']
  Outcome        : visit  range [0, 1]
                   mean  0.1288  (std 0.3350)
  ATE (naive)    : +0.0452  (treated mean - control mean)

Configuration: 1 repeated splits (full dataset, 80/20), DRY_RUN=True


## 4. Evaluation

Run all models across the repeated splits. Per split we:
1. Split the full dataset 80/20 into train / test, stratified on T × Y
2. Tune LightGBM once (regressor + classifier), reuse tuned hyperparameters across all meta-learners
3. Fit S/T/X/R/DR-learners with 5 base models + CausalForestDML + CausalPFN, predict on test set
4. Store AUQC (diagonal-subtracted) and policy gains for later aggregation

**Stratification**: splits are stratified on T × Y — a 4-category interaction of treatment and binary outcome — to ensure each split preserves the proportion of visitors within each treatment arm.

In [ ]:
import time

X_full = pd.DataFrame(all_X_arr, columns=feature_cols)
strat_full = all_T_arr * 2 + all_Y_arr.astype(int)

print(f"Running {N_SPLITS} repeated splits (full dataset n={len(all_T_arr)}, 80/20)...\n")

for split_idx in range(N_SPLITS):
    rs = 42 + split_idx

    # Stratify on T × Y (both are binary, so this gives 4 strata)
    X_train, X_test, T_train, T_test, Y_train, Y_test = train_test_split(
        X_full, all_T_arr, all_Y_arr, test_size=0.2, random_state=rs, stratify=strat_full)
    X_train = X_train.reset_index(drop=True)
    X_test = X_test.reset_index(drop=True)

    ate_test = Y_test[T_test == 1].mean() - Y_test[T_test == 0].mean()
    print(f"\n{'='*60}")
    print(f"Split {split_idx + 1}/{N_SPLITS}  |  test ATE: {ate_test:+.4f}"
          f"  (n={len(T_test)}, treated={T_test.sum()}, control={(T_test==0).sum()})")
    print(f"{'='*60}")

    # ── Arm splits (needed for T- and X-learner tuning) ──────────────────────
    trt_mask  = T_train == 1
    ctrl_mask = T_train == 0
    X_trt, Y_trt = X_train[trt_mask].reset_index(drop=True), Y_train[trt_mask]
    X_ctrl, Y_ctrl = X_train[ctrl_mask].reset_index(drop=True), Y_train[ctrl_mask]


    # ── LightGBM hyperparameter tuning ───────────────────────────────────────────
    print(f"  [LightGBM] Tuning hyperparameters...", end=" ", flush=True)
    t0 = time.time()
    X_with_T        = np.column_stack([X_train, T_train])
    params_s        = tune_lgbm(X_with_T, Y_train, stratify=T_train)  # S-learner outcome (X+T features)
    params_outcome  = tune_lgbm(X_train,  Y_train, stratify=T_train)  # outcome nuisance (R/DR)
    params_prop     = tune_lgbm(X_train,  T_train, classifier=True)   # propensity model
    params_ctrl     = tune_lgbm(X_ctrl,   Y_ctrl)                     # T/X control arm outcome
    params_trt      = tune_lgbm(X_trt,    Y_trt)                      # T/X treated arm outcome
    print(f"  LightGBM tuning: {time.time() - t0:.1f}s")


    # ── Model configs ────────────────────────────────────────────────────────
    # Local shorthands to avoid repeating constructor args across 4 models × 11 roles
    def _lgbm_r(params, seed=42): return LGBMRegressor(random_state=seed, verbose=-1, **params)
    def _lgbm_c(params, seed=42): return LGBMClassifier(random_state=seed, verbose=-1, **params)
    def _tabpfn_r(seed=None):     return TabPFNRegressor(device=device, random_state=rs if seed is None else seed)
    def _tabpfn_c(seed=None):     return TabPFNClassifier(device=device, random_state=rs if seed is None else seed)
    def _tabicl_r(seed=None):     return TabICLRegressor(device=tabicl_device, random_state=rs if seed is None else seed, verbose=False)
    def _tabicl_c(seed=None):     return TabICLClassifier(device=tabicl_device, random_state=rs if seed is None else seed, verbose=False)
    def _tabpfn25_r(seed=None):              return TabPFNRegressor25(device=device, random_state = rs if seed is None else seed)
    def _tabpfn25_c(seed=None):              return TabPFNClassifier25(device=device, random_state = rs if seed is None else seed)

    configs = {
        'LinearRegression': {
            's_model':       LinearRegression(),
            't_models':      (LinearRegression(), LinearRegression()),
            'x_models':      (LinearRegression(), LinearRegression()),
            'x_cate':        (LinearRegression(), LinearRegression()),
            'x_propensity':  LogisticRegression(max_iter=1000, random_state=44),
            'r_model_y':     LinearRegression(),
            'r_model_t':     LogisticRegression(max_iter=1000, random_state=42),
            'r_model_final': LinearRegression(),
            'dr_regression': LinearRegression(),
            'dr_propensity': LogisticRegression(max_iter=1000, random_state=43),
            'dr_final':      LinearRegression(),
        },
        'LightGBM': {
            's_model':       _lgbm_r(params_s),
            # T/X-learner: (models[0]=control arm, models[1]=treated arm) — EconML convention
            't_models':      (_lgbm_r(params_ctrl, 42), _lgbm_r(params_trt, 43)),
            'x_models':      (_lgbm_r(params_ctrl, 42), _lgbm_r(params_trt, 43)),
            'x_cate':        (make_lgbm_final(46), make_lgbm_final(47)),
            'x_propensity':  _lgbm_c(params_prop, 44),
            'r_model_y':     _lgbm_r(params_outcome, 42),
            'r_model_t':     _lgbm_c(params_prop, 43),
            'r_model_final': make_lgbm_final(44),
            'dr_regression': _lgbm_r(params_s, 42),       # tuned on [X,T], matches DRLearner's internal input
            'dr_propensity': _lgbm_c(params_prop, 44),
            'dr_final':      make_lgbm_final(45),
        },
        'TabPFN_v2.5': {
            's_model':       _tabpfn25_r(),
            't_models':      (_tabpfn25_r(), _tabpfn25_r(rs+1)),
            'x_models':      (_tabpfn25_r(), _tabpfn25_r(rs+1)),
            'x_cate':        (_tabpfn25_r(rs+2), _tabpfn25_r(rs+3)),
            'x_propensity':  _tabpfn25_c(),
            'r_model_y':     _tabpfn25_r(),
            'r_model_t':     _tabpfn25_c(),
            'r_model_final': make_lgbm_final(44),
            'dr_regression': _tabpfn25_r(),
            'dr_propensity': _tabpfn25_c(),
            'dr_final':      _tabpfn25_r(rs+2),
        },
        'TabPFN_v2.6': {
            's_model':       _tabpfn_r(),
            't_models':      (_tabpfn_r(), _tabpfn_r(rs+1)),
            'x_models':      (_tabpfn_r(), _tabpfn_r(rs+1)),
            'x_cate':        (_tabpfn_r(rs+2), _tabpfn_r(rs+3)),
            'x_propensity':  _tabpfn_c(),
            'r_model_y':     _tabpfn_r(),
            'r_model_t':     _tabpfn_c(),
            'r_model_final': make_lgbm_final(44),  # TabPFN not suited for residual-on-residual
            'dr_regression': _tabpfn_r(),
            'dr_propensity': _tabpfn_c(),
            'dr_final':      _tabpfn_r(rs+2),
        },
        'TabICL': {
            's_model':       _tabicl_r(),
            # T/X-learner: (models[0]=control arm, models[1]=treated arm) — EconML convention
            't_models':      (_tabicl_r(), _tabicl_r(rs+1)),
            'x_models':      (_tabicl_r(), _tabicl_r(rs+1)),
            'x_cate':        (_tabicl_r(rs+2), _tabicl_r(rs+3)),
            'x_propensity':  _tabicl_c(),
            'r_model_y':     _tabicl_r(),
            'r_model_t':     _tabicl_c(),
            'r_model_final': make_lgbm_final(44),  # TabICL not suited for residual-on-residual
            'dr_regression': _tabicl_r(),
            'dr_propensity': _tabicl_c(),
            'dr_final':      _tabicl_r(rs+2),
        },
    }

    def evaluate(meta, name, te_pred):
        """Compute and store all metrics for one (meta, name, split)."""
        te = np.asarray(te_pred).flatten()
        curve, auqc = get_qini_curve(T_test, Y_test, te)
        results[meta][name]['auqc'].append(auqc)
        results[meta][name]['g1'].append(float(curve[-1]))
        qini_curves[meta][name].append(interpolate_qini_curve(curve, n_grid=N_GRID))
        topk = calculate_topk_policy_metrics(Y_test, T_test, te)
        for k in TOP_K_PERCENTAGES:
            results[meta][name][f'gain_{k}'].append(topk[f'policy_gain_{k}'])


    # ── Evaluation ────────────────────────────────────────────────────────────
    for name, cfg in configs.items():

        # ── S-Learner ───────────────────────────────────────────────────────────────
        print(f"  [{name}] S-learner...", end=" ", flush=True)
        _t0 = time.time()
        sl = SLearner(overall_model=cfg['s_model'])
        sl.fit(Y_train, T_train, X=X_train)
        evaluate('S', name, sl.effect(X_test))
        print(f"done. ({time.time() - _t0:.1f}s)")


        # ── T-Learner ───────────────────────────────────────────────────────────────
        print(f"  [{name}] T-learner...", end=" ", flush=True)
        _t0 = time.time()
        tl = TLearner(models=cfg['t_models'])
        tl.fit(Y_train, T_train, X=X_train)
        evaluate('T', name, tl.effect(X_test))
        print(f"done. ({time.time() - _t0:.1f}s)")


        # ── X-Learner ───────────────────────────────────────────────────────────────
        print(f"  [{name}] X-learner...", end=" ", flush=True)
        _t0 = time.time()
        xl = XLearner(models=cfg['x_models'], cate_models=cfg['x_cate'], propensity_model=cfg['x_propensity'])
        xl.fit(Y_train, T_train, X=X_train)
        evaluate('X', name, xl.effect(X_test))
        print(f"done. ({time.time() - _t0:.1f}s)")


        # ── R-learner (NonParamDML) ──────────────────────────────────────────────
        print(f"  [{name}] R-learner (NonParamDML, cv={NUISANCE_CV})...", end=" ", flush=True)
        _t0 = time.time()
        rl = NonParamDML(
            model_y=cfg['r_model_y'], model_t=cfg['r_model_t'],
            model_final=cfg['r_model_final'], discrete_treatment=True, cv=NUISANCE_CV
        )
        rl.fit(Y_train, T_train, X=X_train)
        evaluate('R', name, rl.effect(X_test))
        print(f"done. ({time.time() - _t0:.1f}s)")


        # ── DR-learner ───────────────────────────────────────────────────────────
        print(f"  [{name}] DR-learner (DRLearner, cv={NUISANCE_CV})...", end=" ", flush=True)
        _t0 = time.time()
        dl = DRLearner(
            model_regression=cfg['dr_regression'],
            model_propensity=cfg['dr_propensity'],
            model_final=cfg['dr_final'],
            min_propensity=0.05,
            cv=NUISANCE_CV
        )
        dl.fit(Y_train, T_train, X=X_train)
        evaluate('DR', name, dl.effect(X_test))
        print(f"done. ({time.time() - _t0:.1f}s)")


    # ── Causal Forest (CausalForestDML) ───────────────────────────────────────
    try:
        print(f"  [CausalForest] DML (cv={NUISANCE_CV})...", end=" ", flush=True)
        _t0 = time.time()
        cf = CausalForestDML(
            model_y=LGBMRegressor(random_state=42, verbose=-1, **params_outcome),
            model_t=LGBMClassifier(random_state=43, verbose=-1, **params_prop),
            discrete_treatment=True,
            cv=NUISANCE_CV,
            n_estimators=200,
            min_samples_leaf=5,
            random_state=rs,
        )
        cf.tune(Y_train, T_train, X=X_train)
        cf.fit(Y_train, T_train, X=X_train)
        evaluate('CF', 'CausalForest', cf.effect(X_test))
        print(f"done. ({time.time() - _t0:.1f}s)")
    except Exception as exc:
        print(f"ERROR\n  CausalForest error on split {split_idx + 1}: {exc}")


    # ── CausalPFN ───────────────────────────────────────────────────────────────
    try:
        print(f"  [CausalPFN]...", end=" ", flush=True)
        _t0 = time.time()
        cpfn = CATEEstimator(device=causalpfn_device, verbose=False)

        X_cpfn_train = np.asarray(X_train, dtype=np.float32)
        T_cpfn_train = np.asarray(T_train, dtype=np.float32).ravel()
        Y_cpfn_train = np.asarray(Y_train, dtype=np.float32).ravel()
        X_cpfn_test  = np.asarray(X_test, dtype=np.float32)

        cpfn.fit(X_cpfn_train, T_cpfn_train, Y_cpfn_train)
        te = cpfn.estimate_cate(X_cpfn_test)

        if "torch" in str(type(te)):
            te = te.detach().cpu().numpy()

        te = np.asarray(te, dtype=np.float32).reshape(-1)
        evaluate('CausalPFN', 'CausalPFN', te)
        print(f"done. ({time.time() - _t0:.1f}s)")
    except Exception as exc:
        print(f"ERROR\n  CausalPFN error on split {split_idx + 1}: {exc}")

print(f"\nDone. {N_SPLITS} splits evaluated.")

Running 1 repeated splits (full dataset n=42693, 80/20)...


Split 1/1  |  test ATE: +0.0454  (n=8539, treated=4278, control=4261)
  [LightGBM] Tuning hyperparameters... 

## Save Results to Disk

In [ ]:
# --- Save results to disk ---
_save_path = os.path.join(RESULTS_DIR, f'hillstrom_{arm_tag}_results.pkl')
with open(_save_path, 'wb') as f:
    pickle.dump({
        'results': {m: {n: dict(d) for n, d in v.items()} for m, v in results.items()},
        'qini_curves': {m: {n: list(c) for n, c in v.items()} for m, v in qini_curves.items()},
        'N_SPLITS': N_SPLITS,
        'N_GRID': N_GRID,
        'META_LEARNERS': META_LEARNERS,
        'BASE_MODELS': BASE_MODELS,
        'TOP_K_PERCENTAGES': TOP_K_PERCENTAGES,
        'OUTCOME_COL': OUTCOME_COL,
        'TREATMENT_ARM': TREATMENT_ARM,
        'CONTROL_ARM': CONTROL_ARM,
        'arm_tag': arm_tag,
        'n_total': n_total,
    }, f)
print(f"Results saved to {_save_path}")

## 5. Results

We aggregate per-split metrics across the 10 repetitions and report mean ± SE.

**Reported metrics**:
- AUQC (area between Qini curve and its diagonal, unitless): mean ± SE, 95% CI, significant if CI excludes zero. Expected value under a random CATE ranking is 0 by construction.
- Gain@10/20/30/40%: mean ± SE (already a delta: policy value minus random targeting baseline, in visit rate units)

In [ ]:
# # --- Load results from disk (run this cell instead of the evaluation loop when resuming) ---
# with open('/Users/jensbrehmen/Documents/KUL/Thesis/Notebooks/Final results Hillstorm/hillstrom_Hill2_results.pkl', 'rb') as f:
#     _data = pickle.load(f)
# results = defaultdict(lambda: defaultdict(lambda: defaultdict(list)), {
#     m: defaultdict(lambda: defaultdict(list), {n: defaultdict(list, d) for n, d in v.items()})
#     for m, v in _data['results'].items()
# })
# qini_curves = defaultdict(lambda: defaultdict(list), {
#     m: defaultdict(list, v) for m, v in _data['qini_curves'].items()
# })
# N_SPLITS          = _data['N_SPLITS']
# N_GRID            = _data['N_GRID']
# META_LEARNERS     = _data['META_LEARNERS']
# BASE_MODELS       = _data['BASE_MODELS']
# TOP_K_PERCENTAGES = _data['TOP_K_PERCENTAGES']
# OUTCOME_COL       = _data['OUTCOME_COL']
# TREATMENT_ARM     = _data['TREATMENT_ARM']
# CONTROL_ARM       = _data['CONTROL_ARM']
# arm_tag           = _data['arm_tag']
# print(f"Loaded results: {N_SPLITS} splits, arm={arm_tag}")

Loaded results: 10 splits, arm=Hill2


In [ ]:
# --- Aggregate ---
STANDALONE_MODELS = {'CausalPFN': 'CausalPFN', 'CF': 'CausalForest'}

rows = []
for meta in META_LEARNERS:
    model_list = [STANDALONE_MODELS[meta]] if meta in STANDALONE_MODELS else BASE_MODELS
    for name in model_list:
        d = results[meta][name]
        if not d['auqc']:
            continue
        row = {'Meta': meta, 'Base': name}

        def agg(vals, label):
            arr = np.array(vals, dtype=float)
            valid = arr[~np.isnan(arr)]
            nv = len(valid)
            row[f'{label} Mean'] = np.nanmean(arr) if nv > 0 else np.nan
            row[f'{label} SE'] = np.nanstd(valid, ddof=1) / np.sqrt(nv) if nv > 1 else np.nan
            row[f'{label} N'] = nv
            if label == 'AUQC' and nv > 1:
                mean = np.nanmean(valid)
                se = np.nanstd(valid, ddof=1) / np.sqrt(nv)
                tcrit = st.t.ppf(0.975, df=nv - 1)
                ci_lo, ci_hi = mean - tcrit * se, mean + tcrit * se
                row[f'{label} CI'] = f"[{ci_lo:.4f}, {ci_hi:.4f}]"
                row[f'{label} Sig'] = '*' if ci_lo > 0 or ci_hi < 0 else ''
            return row

        agg(d['auqc'], 'AUQC')
        for k in TOP_K_PERCENTAGES:
            agg(d[f'gain_{k}'], f'Gain@{k}%')

        rows.append(row)

df = pd.DataFrame(rows)

# Sort by AUQC Mean descending (best first)
df = df.sort_values('AUQC Mean', ascending=False).reset_index(drop=True)

# --- Display: AUQC ---
print("=" * 110)
print(f"RESULTS ACROSS {N_SPLITS} SPLITS (mean ± SE) — ranked by AUQC")
print("=" * 110)
print("AUQC is diagonal-subtracted: expected value under random model is 0 by construction.")

auqc_cols = ['Meta', 'Base', 'AUQC Mean', 'AUQC SE', 'AUQC CI', 'AUQC Sig']
auqc_cols = [c for c in auqc_cols if c in df.columns]
print("\n--- AUQC (diagonal-subtracted) ---")
print(df[auqc_cols].to_string(index=False, float_format='%.4f'))

# --- Display: Policy Gains ---
for k in TOP_K_PERCENTAGES:
    gain_cols = ['Meta', 'Base', f'Gain@{k}% Mean', f'Gain@{k}% SE']
    existing = [c for c in gain_cols if c in df.columns]
    df_k = df.sort_values(f'Gain@{k}% Mean', ascending=False).reset_index(drop=True)
    print(f"\n--- Policy Gain @{k}% (ranked) ---")
    print(df_k[existing].to_string(index=False, float_format='%.4f'))

# --- Bar charts ---
n_bars = 1 + len(TOP_K_PERCENTAGES)
fig, axes = plt.subplots(1, n_bars, figsize=(6 * n_bars, 6))

# AUQC bar chart
ax = axes[0]
plot_data = df.pivot(index='Base', columns='Meta', values='AUQC Mean')
plot_err = df.pivot(index='Base', columns='Meta', values='AUQC SE')
plot_data.plot(kind='bar', yerr=plot_err, ax=ax, capsize=3, rot=0)
ax.set_title('AUQC (diagonal-subtracted)', fontsize=10)
ax.set_ylabel('Mean ± SE')
ax.axhline(y=0, color='red', linestyle='--', linewidth=1.5)
ax.grid(True, alpha=0.3, axis='y')
ax.legend(fontsize=7, loc='best')

# Gain@k% bars
for i, k in enumerate(TOP_K_PERCENTAGES):
    ax = axes[i + 1]
    plot_data = df.pivot(index='Base', columns='Meta', values=f'Gain@{k}% Mean')
    plot_err = df.pivot(index='Base', columns='Meta', values=f'Gain@{k}% SE')
    plot_data.plot(kind='bar', yerr=plot_err, ax=ax, capsize=3, rot=0)
    ax.set_title(f'Gain@{k}% (policy − random)', fontsize=10)
    ax.set_ylabel('Mean ± SE (visit rate)')
    ax.axhline(y=0, color='red', linestyle='--', linewidth=1.5)
    ax.grid(True, alpha=0.3, axis='y')
    ax.legend(fontsize=7, loc='best')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'bar_charts.pdf'), bbox_inches='tight')
plt.show()

# --- Qini Curves: 2x4 grid ---
x_grid = np.linspace(0, 1, N_GRID)
colors = {'LinearRegression': 'blue', 'LightGBM': 'green', 'TabPFN_v2.5': 'red', 'TabPFN_v2.6': 'orange',
          'TabICL': 'teal', 'CausalForest': 'brown', 'CausalPFN': 'purple'}

# --- Analytical diagonal baseline band ---
# G(1) is a property of (T_test, Y_test) only — identical across all models on the same split.
# Use stored g1 values from S/LightGBM (always present) so the band reflects only
# split-to-split variance in G(1), not model variance.
_g1_per_split = results['S']['LightGBM']['g1']
_all_diag = np.array([x_grid * g1 for g1 in _g1_per_split])
diag_mean = _all_diag.mean(axis=0)
diag_se = _all_diag.std(axis=0) / np.sqrt(len(_all_diag))

fig, axes = plt.subplots(2, 4, figsize=(24, 12))
axes = axes.flatten()

for idx, meta in enumerate(['S', 'T', 'X', 'R', 'DR']):
    ax = axes[idx]
    for base in BASE_MODELS:
        curves_arr = qini_curves[meta][base]
        if curves_arr:
            c = np.array(curves_arr)
            m = c.mean(axis=0)
            se = c.std(axis=0) / np.sqrt(len(c))
            auqc_mean = np.mean(results[meta][base]['auqc'])
            ax.plot(x_grid, m, color=colors[base], linewidth=2,
                    label=f'{base} (AUQC={auqc_mean:.4f})')
            ax.fill_between(x_grid, m - se, m + se, color=colors[base], alpha=0.12)
    ax.fill_between(x_grid, diag_mean - diag_se, diag_mean + diag_se, color='gray', alpha=0.2, label='Diagonal ± 1 SE')
    ax.plot(x_grid, diag_mean, 'k--', linewidth=1, alpha=0.5)
    ax.set_title(f'{meta}-Learner')
    ax.set_xlabel('Fraction targeted')
    ax.set_ylabel('Cumulative incremental visit rate')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

ax = axes[5]
curves_arr = qini_curves['CF']['CausalForest']
if curves_arr:
    c = np.array(curves_arr)
    m = c.mean(axis=0)
    se = c.std(axis=0) / np.sqrt(len(c))
    auqc_mean = np.mean(results['CF']['CausalForest']['auqc'])
    ax.plot(x_grid, m, color=colors['CausalForest'], linewidth=2,
            label=f'CausalForest (AUQC={auqc_mean:.4f})')
    ax.fill_between(x_grid, m - se, m + se, color=colors['CausalForest'], alpha=0.12)
ax.fill_between(x_grid, diag_mean - diag_se, diag_mean + diag_se, color='gray', alpha=0.2, label='Diagonal ± 1 SE')
ax.plot(x_grid, diag_mean, 'k--', linewidth=1, alpha=0.5)
ax.set_title('Causal Forest (DML)')
ax.set_xlabel('Fraction targeted')
ax.set_ylabel('Cumulative incremental visit rate')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

ax = axes[6]
curves_arr = qini_curves['CausalPFN']['CausalPFN']
if curves_arr:
    c = np.array(curves_arr)
    m = c.mean(axis=0)
    se = c.std(axis=0) / np.sqrt(len(c))
    auqc_mean = np.mean(results['CausalPFN']['CausalPFN']['auqc'])
    ax.plot(x_grid, m, color=colors['CausalPFN'], linewidth=2,
            label=f'CausalPFN (AUQC={auqc_mean:.4f})')
    ax.fill_between(x_grid, m - se, m + se, color=colors['CausalPFN'], alpha=0.12)
ax.fill_between(x_grid, diag_mean - diag_se, diag_mean + diag_se, color='gray', alpha=0.2, label='Diagonal ± 1 SE')
ax.plot(x_grid, diag_mean, 'k--', linewidth=1, alpha=0.5)
ax.set_title('CausalPFN')
ax.set_xlabel('Fraction targeted')
ax.set_ylabel('Cumulative incremental visit rate')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

axes[7].set_visible(False)

plt.suptitle(f'Qini Curves ({N_SPLITS} splits, mean ± 1 SE)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'qini_curves.pdf'), bbox_inches='tight')
plt.show()
